# POC: Instantiate Points object as pydantic class

As part of the future refactoring of geoh5py, this spike explores creating an entity as a pydantic class.

Goals:

- Direct access of attributes and data from the geoh5 dataset
- Validation of attributes as pydantic model fields
- Lazy loading of large arrays such as vertices, cells, and data
- Instantiation of the class without the need for a parent workspace

## Imports



In [ ]:
import pickle
from uuid import uuid4

import numpy as np
from pydantic import ValidationError

from geoh5py_pydantic import VERTICES_DTYPE, CallableArraySource, PointsModel

## Instantiate without a workspace

Create a Points-like entity directly from attributes and vertices without creating a Workspace.

In [ ]:
# PointsModel can be created directly from plain coordinates.
example_array = np.array(
    [
        [0.0, 0.0, 0.0],
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
    ]
)

points = PointsModel(
    name="Standalone points",
    vertices=example_array,
)

# Can pickle the PointsModel:

with open("points.pkl", "wb") as file:
    pickle.dump(points, file)

with open("points.pkl", "rb") as file:
    points_loaded = pickle.load(file)

points_loaded

In [ ]:
points

In [ ]:
# Entity-style attributes and computed geometry are available directly.
points.uid, points.name, points.n_vertices

In [ ]:
# Vertices are exposed as a normal (n, 3) float array for convenient use.
points.vertices_array

In [ ]:
# Extent is derived from the vertex coordinates.
points.extent

## Assignment validation

Pydantic assignment validation means changing model fields should rerun the same validation rules.

In [ ]:
# Replacing vertices with another valid array succeeds and updates derived values.
points.vertices = np.array([[10.0, 11.0, 12.0], [13.0, 14.0, 15.0]])
points.vertices_array

In [ ]:
# Invalid shapes cause pydantic ValidationError messages.
try:
    PointsModel(vertices=np.r_[1.0, 2.0, 3.0])
except ValidationError as error:
    print(error)

## Lazy array

This simulates a future geoh5 adapter where large arrays are not loaded until code actually asks for them.

In [ ]:
# CallableArraySource simulates some kind of IO adapter that fetches vertices by uid/key.
# PointsModel should be able to represent a Points object but delay loading the actual vertex array until later
uid = uuid4()
calls = []


def fetcher(entity_uid, key):
    # if asked for an array belonging to entity_uid and named key,
    # record that this was called then return this fake vertices array
    calls.append((str(entity_uid), key))
    return np.array(
        [
            [100.0, 200.0, 300.0],
            [101.0, 201.0, 301.0],
        ]
    )


source = CallableArraySource(fetcher)

lazy_points = PointsModel.from_array_source(source, uid, name="Lazy points")

lazy_points.vertices, lazy_points.vertices.is_loaded, calls
# Should show that the vertices array is not loaded yet, and the fetcher has not been called.

In [ ]:
# Accessing n_vertices forces the LazyArray to load and validate its data.
lazy_points.n_vertices, lazy_points.vertices.is_loaded, calls

In [ ]:
# Subsequent access reuses the cached loaded array.
lazy_points.vertices_array

## HDF5 serialization categories

The entity now owns a nested Attributes model and EntityType model. Larger or specially formatted values remain datasets.

In [ ]:
# Create a fresh lazy model so this section can show exactly when serialization loads vertices.
example_uid = uuid4()  # random uid
serialization_calls = []


def serialization_fetcher(entity_uid, key):
    # Record each simulated IO request so repeated or premature loads are easy to spot.
    serialization_calls.append((str(entity_uid), key))
    return np.array([[10.0, 20.0, 30.0], [11.0, 21.0, 31.0]])


serialization_points = PointsModel.from_array_source(
    CallableArraySource(
        serialization_fetcher
    ),  # callable array source allows us to see when the array is loaded
    example_uid,
    name="Serializable points",
    metadata={"purpose": "serialization example"},
)

serialization_points.vertices.is_loaded, serialization_calls
# should not be loaded yet, and no simulated IO should have occurred

In [ ]:
# Scalar fields are stored in the nested Attributes model, while type identity and
# HDF5 placement belong to EntityType. Dataset mappings remain on the entity for now.
(
    serialization_points.attributes,
    serialization_points.entity_type,
    serialization_points.entity_type.h5_collection,
    serialization_points.entity_type.h5_type_collection,
    serialization_points.dataset_map,
)

In [ ]:
# Pydantic aliases convert the nested Attributes fields directly to their HDF5 names.
# This only reads small scalar values, so it does not load the lazy vertex array.

h5_attributes = serialization_points.attributes.model_dump(
    by_alias=True,
    exclude_none=True,
)

h5_attributes

In [ ]:
# The empty call list confirms that no simulated IO occurred.
serialization_points.vertices.is_loaded, serialization_calls

In [ ]:
# Preparing datasets requires the actual vertices, so this is where lazy loading occurs.
# The vertices field serializer also converts the plain n,3 array to geoh5's structured dtype here
h5_datasets = serialization_points.h5_datasets()
h5_vertices = h5_datasets["Vertices"]

(
    h5_datasets,
    h5_vertices.dtype,
    h5_vertices.dtype == VERTICES_DTYPE,
    h5_vertices.view("<f8").reshape((-1, 3)),
    serialization_points.vertices.is_loaded,
    serialization_calls,
)

In [ ]:
# Calling the dataset dump again reuses the cached LazyArray value rather than fetching twice.
serialization_points.h5_datasets()
len(serialization_calls), serialization_calls

## Generic geoh5 writer

The new writer converts a workspace-free model into a generic payload, then writes that payload directly with h5py

Right now, file initialization is still separate. `Workspace.create()` is used below only to create an empty project and Root structure that `Geoh5Writer` can write into.

In [ ]:
# A payload contains everything the generic writer needs, without depending on PointsModel itself.
# Creating it prepares attributes, datasets, type information, and the optional parent UID.
from geoh5py_pydantic.serialization import Geoh5EntityPayload, Geoh5Writer


payload = Geoh5EntityPayload.from_model(serialization_points)

(
    payload.collection,
    payload.type_collection,
    payload.uid,
    payload.type_uid,
    sorted(payload.attributes),
    sorted(payload.datasets),
)

In [ ]:
# Geoh5Writer currently expects an initialized geoh5 file.
# temporarily, we use Workspace just to create only the project/Root structure.
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import h5py

from geoh5py.workspace import Workspace


writer_temp_dir = TemporaryDirectory()
writer_path = Path(writer_temp_dir.name) / "pydantic_writer.geoh5"

with Workspace.create(writer_path):
    pass

writer_path

In [ ]:
# write(model) builds the same payload internally and writes directly to HDF5.
# Values copied into writer_summary remain usable after the h5py file is closed.
with h5py.File(writer_path, "r+") as h5file:
    writer = Geoh5Writer(h5file)
    written_group = writer.write(serialization_points)

    uid_text = writer.format_uuid(serialization_points.uid)
    type_uid_text = writer.format_uuid(serialization_points.type_uid)
    root_link = writer.project["Root"]["Objects"][uid_text]
    type_group = writer.project["Types"]["Object types"][type_uid_text]

    writer_summary = {
        "canonical path": written_group.name,
        "attribute names": sorted(written_group.attrs),
        "dataset/group names": sorted(written_group),
        "vertices dtype": written_group["Vertices"].dtype,
        "metadata": json.loads(written_group["Metadata"][0]),
        "Root link points to same group": root_link.id == written_group.id,
        "Type link points to shared type": written_group["Type"].id == type_group.id,
    }

writer_summary, len(serialization_calls), serialization_calls

In [ ]:
# Reopening with existing geoh5py is a compatibility check, not part of the new write path.
# If this succeeds, the direct HDF5 layout is readable by current geoh5py.
with Workspace(writer_path) as reader_workspace:
    recovered = reader_workspace.get_entity(serialization_points.uid)[0]
    recovered_summary = (
        type(recovered).__name__,
        recovered.uid,
        recovered.name,
        recovered.entity_type.uid,
        recovered.vertices,
        recovered.metadata,
    )

recovered_summary

In [ ]:
# Remove the temporary demonstration file now that both write and read checks are complete.
writer_temp_dir.cleanup()

## Legacy adapter

A way for existing geoh5py Points objects to be adapted to the pydantic model.
If the plan is just to swap from the current geoh5py setup to this new version, this adapter probably won't be needed
though it could be useful for testing and comparison.

In [ ]:
# Existing geoh5py Points can be adapted to the pydantic model for comparison.
from geoh5py.objects import Points
from geoh5py.workspace import Workspace


workspace = Workspace()
legacy = Points.create(
    workspace,
    name="Legacy points",
    vertices=np.array([[1.0, 1.0, 1.0], [2.0, 2.0, 2.0]]),
)

adapted = PointsModel.from_legacy_points(legacy)
type(legacy), type(adapted), adapted.name, adapted.vertices_array